In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/file_234.txt
/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/file_470.txt
/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/file_1303.txt
/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/file_425.txt
/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/file_1709.txt
/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/file_146.txt
/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/file_2872.txt
/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/file_1520.txt
/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/file_637.txt
/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/file_3400.txt
/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/file_2541.txt
/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/file_4217.txt
/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/file_1861.txt
/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/file_2122.txt
/kaggle/input/datasets/is

In [2]:
!git clone https://github.com/Law-AI/semantic-segmentation.git

Cloning into 'semantic-segmentation'...
remote: Enumerating objects: 712, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 712 (delta 37), reused 31 (delta 31), pack-reused 666 (from 1)
Receiving objects: 100% (712/712), 6.39 MiB | 18.37 MiB/s, done.
Resolving deltas: 100% (242/242), done.


In [4]:
cd semantic-segmentation

/kaggle/working/semantic-segmentation


In [6]:
!pip install datasets transformers huggingface_hub

-- token for dataset access from HuggingFace --

In [10]:
from datasets import load_dataset
datasets = load_dataset("Exploration-Lab/IL-TUR", "pcr")

README.md: 0.00B [00:00, ?B/s]

pcr/train_candidates-00000-of-00001.parq(…):   0%|          | 0.00/77.0M [00:00<?, ?B/s]

pcr/dev_candidates-00000-of-00001.parque(…):   0%|          | 0.00/25.1M [00:00<?, ?B/s]

pcr/test_candidates-00000-of-00001.parqu(…):   0%|          | 0.00/34.2M [00:00<?, ?B/s]

pcr/train_queries-00000-of-00001.parquet:   0%|          | 0.00/16.0M [00:00<?, ?B/s]

...

Generating dev_candidates split:   0%|          | 0/1023 [00:00<?, ? examples/s]

Generating test_candidates split:   0%|          | 0/1727 [00:00<?, ? examples/s]

Generating train_queries split:   0%|          | 0/827 [00:00<?, ? examples/s]

Generating dev_queries split:   0%|          | 0/118 [00:00<?, ? examples/s]

Generating test_queries split:   0%|          | 0/237 [00:00<?, ? examples/s]

In [11]:
print(datasets)

DatasetDict({
    train_candidates: Dataset({
        features: ['id', 'text', 'relevant_candidates'],
        num_rows: 4320
    })
    dev_candidates: Dataset({
        features: ['id', 'text', 'relevant_candidates'],
        num_rows: 1023
    })
    test_candidates: Dataset({
        features: ['id', 'text', 'relevant_candidates'],
        num_rows: 1727
    })
    train_queries: Dataset({
        features: ['id', 'text', 'relevant_candidates'],
        num_rows: 827
    })
    dev_queries: Dataset({
        features: ['id', 'text', 'relevant_candidates'],
        num_rows: 118
    })
    test_queries: Dataset({
        features: ['id', 'text', 'relevant_candidates'],
        num_rows: 237
    })
})


In [12]:
import os

files = [f for f in os.listdir('/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/') 
         if f.endswith('.txt')]

print(f"Total .txt files: {len(files)}")
print(f"Sample: {files[:5]}")

Total .txt files: 4984
Sample: ['file_234.txt', 'file_470.txt', 'file_1303.txt', 'file_425.txt', 'file_1709.txt']


In [13]:
import os
import shutil

train_path = '/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/'

labels = set()
files = [f for f in os.listdir(train_path) if f.endswith('.txt')]

for fname in files:
    with open(f'{train_path}/{fname}') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                labels.add(parts[1])

print(f"Total docs: {len(files)}")
print(f"Labels found: {labels}")

Total docs: 4984
Labels found: {'Issue', 'Arguments of Petitioner', 'Reasoning', 'Decision', 'Facts', 'Arguments of Respondent', 'None'}


In [14]:
import os
import shutil

src_path = "/kaggle/input/datasets/ishitarsk/legalseg-dataset/train/"
dest_path = "/kaggle/working/semantic-segmentation/data/text/"

# 1. Ensure destination exists
os.makedirs(dest_path, exist_ok=True)

# 2. Delete old files in destination (IMPORTANT: overwrite behavior)
for f in os.listdir(dest_path):
    file_path = os.path.join(dest_path, f)
    if os.path.isfile(file_path):
        os.remove(file_path)

# 3. Copy only .txt files
files = [f for f in os.listdir(src_path) if f.endswith(".txt")]

for fname in files:
    shutil.copy(
        os.path.join(src_path, fname),
        os.path.join(dest_path, fname)
    )

print(f"Replaced dataset: {len(files)} files copied into {dest_path}")

Replaced dataset: 4984 files copied into /kaggle/working/semantic-segmentation/data/text/


In [16]:
file_path = "/kaggle/working/semantic-segmentation/train.py"

with open(file_path, "w") as f:
    f.write("""
import torch
import time
import json
import random
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report


# Randomly shuffle the data and divide into batches
def batchify(x, y, batch_size):
    idx = list(range(len(x)))
    random.shuffle(idx)
    
    # convert to numpy array for ease of indexing
    x = np.array(x, dtype=object)[idx]
    y = np.array(y, dtype=object)[idx]
    
    i = 0
    while i < len(x):
        j = min(i + batch_size, len(x))
        
        batch_idx = idx[i : j]
        batch_x = x[i : j]
        batch_y = y[i : j]
        
        yield batch_idx, batch_x, batch_y
        
        i = j

def train_step(model, opt, x, y, batch_size):
    model.train()
    
    total_loss = 0
    y_pred = []
    y_gold = []
    idx = []
    
    for i, (batch_idx, batch_x, batch_y) in enumerate(batchify(x, y, batch_size)):
        pred = model(batch_x)
        loss = model._loss(batch_y)        

        opt.zero_grad()
        loss.backward()
        opt.step()
        
        total_loss += loss.item()
     
        y_pred.extend(pred)
        y_gold.extend(batch_y)
        idx.extend(batch_idx)
        
    assert len(sum(y, [])) == len(sum(y_pred, [])), "Mismatch in predicted"
    
    return total_loss / (i + 1), idx, y_gold, y_pred


def val_step(model, x, y, batch_size):
    model.eval()   
    total_loss = 0
    y_pred = []
    y_gold = []
    idx = []
    
    with torch.no_grad():   
        for i, (batch_idx, batch_x, batch_y) in enumerate(batchify(x, y, batch_size)):
            pred = model(batch_x)
            loss = model._loss(batch_y)
               
            total_loss += loss.item()
         
            y_pred.extend(pred)
            y_gold.extend(batch_y)
            idx.extend(batch_idx)
        
    assert len(sum(y, [])) == len(sum(y_pred, [])), "Mismatch in predicted"
    
    return total_loss / (i + 1), idx, y_gold, y_pred


def infer_step(model, x):
    model.eval()
    y_pred = model(x)
    return y_pred    


def statistics(data_state, tag2idx):
    idx, gold, pred = data_state['idx'], data_state['gold'], data_state['pred']
    
    rev_tag2idx = {v: k for k, v in tag2idx.items()}
    tags = [rev_tag2idx[i] for i in range(len(tag2idx)) if rev_tag2idx[i] not in ['<start>', '<end>', '<pad>']]
    
    gold = sum(gold, [])
    pred = sum(pred, [])
    
    print(classification_report(gold, pred, target_names = tags, digits = 3))


def learn(model, x, y, tag2idx, val_fold, args):
    indices = list(range(len(x)))
    random.shuffle(indices)

    split = int(0.9 * len(x))

    train_idx = indices[:split]
    val_idx = indices[split:]
    
    train_x = [x[i] for i in train_idx]
    train_y = [y[i] for i in train_idx]
    
    val_x = [x[i] for i in val_idx]
    val_y = [y[i] for i in val_idx]
    
    opt = torch.optim.Adam(model.parameters(), lr = args.lr, weight_decay = args.reg)
    
    print("{0:>7}  {1:>10}  {2:>6}  {3:>10}  {4:>6}".format('EPOCH', 'Tr_LOSS', 'Tr_F1', 'Val_LOSS', 'Val_F1'))
    print("-----------------------------------------------------------")
    
    best_val_f1 = 0.0
    
    model_state = {}
    data_state = {}
    
    start_time = time.time()
    
    for epoch in range(1, args.epochs + 1):

        train_loss, train_idx, train_gold, train_pred = train_step(model, opt, train_x, train_y, args.batch_size)
        val_loss, val_idx, val_gold, val_pred = val_step(model, val_x, val_y, args.batch_size)

        train_f1 = f1_score(sum(train_gold, []), sum(train_pred, []), average = 'macro')
        val_f1 = f1_score(sum(val_gold, []), sum(val_pred, []), average = 'macro')

        if epoch % args.print_every == 0:
            print("{0:7d}  {1:10.3f}  {2:6.3f}  {3:10.3f}  {4:6.3f}".format(epoch, train_loss, train_f1, val_loss, val_f1))

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            model_state = {
                'epoch': epoch,
                'arch': model,
                'name': model.__class__.__name__,
                'state_dict': model.state_dict(),
                'best_f1': val_f1,
                'optimizer': opt.state_dict()
            }
            data_state = {
                'idx': val_idx,
                'loss': val_loss,
                'gold': val_gold,
                'pred': val_pred
            }
            
    end_time = time.time()
    
    print("Dumping model and data ...", end=' ')
    
    torch.save(model_state, args.save_path + 'model_state.tar')
    
    with open(args.save_path + 'data_state.json', 'w') as fp:
        json.dump(data_state, fp)
    
    print("Done")    

    print('Time taken:', int(end_time - start_time), 'secs')

    statistics(data_state, tag2idx)
""")

In [17]:
file_path = "/kaggle/working/semantic-segmentation/model/submodels.py"

with open(file_path, "w") as f:
    f.write("""
import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_sequence, pack_padded_sequence, pad_packed_sequence

SEED = 42

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)


    # Sentences are represented as a sequence of word tokens. These sequences are converted into fixed-length sentence embeddings by passing them through a Bi-LSTM. The hidden embedding at the last word is considered to represent the entire sentence.   

class LSTM_Sentence_Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, drop = 0.5, device = 'cuda'):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim // 2, bidirectional = True, batch_first = True)
        self.dropout = nn.Dropout(drop)
        self.hidden = None
        self.device = device        
        
    def init_hidden(self, batch_size):
        return (torch.randn(2, batch_size, self.hidden_dim // 2).to(self.device), torch.randn(2, batch_size, self.hidden_dim // 2).to(self.device))
    
    def forward(self, sentences, sent_lengths):
        ## sentences: tensor[batch_size, max_sent_len]
        ## sent_lengths: list[batch_size]        
        
        # initialize hidden state
        batch_size = sentences.shape[0]
        self.hidden = self.init_hidden(batch_size)
        
        # convert word tokens to word embeddings
        ## tensor[batch_size, max_sent_len] --> tensor[batch_size, max_sent_len, emb_dim] 
        x = self.emb(sentences)
        x = self.dropout(x)
        
        # generate sentence embedding from sequence of word embeddings
        ## tensor[batch_size, max_sent_len, emb_dim] --> tensor[2, batch_size, hidden_dim // 2]
        x = nn.utils.rnn.pack_padded_sequence(x, list(sent_lengths), batch_first = True, enforce_sorted = False)
        _, (x, _) = self.lstm(x, self.hidden)

        ## tensor[2, batch_size, hidden_dim // 2] --> tensor[batch_size, hidden_dim]        
        x = x.permute(1, 0, 2).contiguous().view(batch_size, -1)
        return x
    


    # A Bi-LSTM is used to generate feature vectors for each sentence from the sentence embeddings. The feature vectors are actually context-aware sentence embeddings. These are then fed to a feed-forward network to obtain emission scores for each class at each sentence.

class LSTM_Emitter(nn.Module):
    def __init__(self, n_tags, emb_dim, hidden_dim, drop = 0.5, device = 'cuda'):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        
        self.lstm = nn.LSTM(emb_dim, hidden_dim // 2, bidirectional = True, batch_first = True)
        self.dropout = nn.Dropout(drop)
        self.hidden2tag = nn.Linear(hidden_dim, n_tags)
        self.hidden = None
        self.device = device
        
    def init_hidden(self, batch_size):
        return (torch.randn(2, batch_size, self.hidden_dim // 2).to(self.device), torch.randn(2, batch_size, self.hidden_dim // 2).to(self.device))
    
    def forward(self, sequences):
        ## sequences: tensor[batch_size, max_seq_len, emb_dim]
        
        # initialize hidden state
        self.hidden = self.init_hidden(sequences.shape[0])
        
        # generate context-aware sentence embeddings (feature vectors)
        ## tensor[batch_size, max_seq_len, emb_dim] --> tensor[batch_size, max_seq_len, hidden_dim]
        x, self.hidden = self.lstm(sequences, self.hidden)
        x = self.dropout(x)
        
        # generate emission scores for each class at each sentence
        # tensor[batch_size, max_seq_len, hidden_dim] --> tensor[batch_size, max_seq_len, n_tags]
        x = self.hidden2tag(x)
        return x
    

    # A linear-chain CRF is fed with the emission scores at each sentence, and it finds out the optimal sequence of tags by learning the transition scores.

class CRF(nn.Module):    
    def __init__(self, n_tags, sos_tag_idx, eos_tag_idx, pad_tag_idx = None, device = 'cpu'):
        super().__init__()
        self.n_tags = n_tags
        self.device = device
        self.SOS_TAG_IDX = sos_tag_idx
        self.EOS_TAG_IDX = eos_tag_idx
        self.PAD_TAG_IDX = pad_tag_idx
        
        self.transitions = nn.Parameter(torch.empty(self.n_tags, self.n_tags))
        self.init_weights()
        
    def init_weights(self):
        # initialize transitions from random uniform distribution between -0.1 and 0.1
        nn.init.uniform_(self.transitions, -0.1, 0.1)
        
        # enforce constraints (rows = from, cols = to) with a big negative number.
        # exp(-1000000) ~ 0
        
        # no transitions to SOS
        self.transitions.data[:, self.SOS_TAG_IDX] = -1000000.0
        # no transition from EOS
        self.transitions.data[self.EOS_TAG_IDX, :] = -1000000.0
        
        if self.PAD_TAG_IDX is not None:
            # no transitions from pad except to pad
            self.transitions.data[self.PAD_TAG_IDX, :] = -1000000.0
            self.transitions.data[:, self.PAD_TAG_IDX] = -1000000.0
            # transitions allowed from end and pad to pad
            self.transitions.data[self.PAD_TAG_IDX, self.EOS_TAG_IDX] = 0.0
            self.transitions.data[self.PAD_TAG_IDX, self.PAD_TAG_IDX] = 0.0
            
    def forward(self, emissions, tags, mask = None):
        ## emissions: tensor[batch_size, seq_len, n_tags]
        ## tags: tensor[batch_size, seq_len]
        ## mask: tensor[batch_size, seq_len], indicates valid positions (0 for pad)
        return -self.log_likelihood(emissions, tags, mask = mask)
    
    def log_likelihood(self, emissions, tags, mask = None):                   
        if mask is None:
            mask = torch.ones(emissions.shape[:2])
            
        scores = self._compute_scores(emissions, tags, mask = mask)
        partition = self._compute_log_partition(emissions, mask = mask)
        return torch.sum(scores - partition)
    
    # find out the optimal tag sequence using Viterbi Decoding Algorithm
    def decode(self, emissions, mask = None):      
        if mask is None:
            mask = torch.ones(emissions.shape[:2])
            
        scores, sequences = self._viterbi_decode(emissions, mask)
        return scores, sequences
    
    def _compute_scores(self, emissions, tags, mask):
        batch_size, seq_len = tags.shape
        scores = torch.zeros(batch_size).to(self.device)
        
        # save first and last tags for later
        first_tags = tags[:, 0]
        last_valid_idx = mask.int().sum(1) - 1
        last_tags = tags.gather(1, last_valid_idx.unsqueeze(1)).squeeze()
        
        # add transition from SOS to first tags for each sample in batch
        t_scores = self.transitions[self.SOS_TAG_IDX, first_tags]
        
        # add emission scores of the first tag for each sample in batch
        e_scores = emissions[:, 0].gather(1, first_tags.unsqueeze(1)).squeeze()
        scores += e_scores + t_scores
        
        # repeat for every remaining word
        for i in range(1, seq_len):
            
            is_valid = mask[:, i]
            prev_tags = tags[:, i - 1]
            curr_tags = tags[:, i]
            
            e_scores = emissions[:, i].gather(1, curr_tags.unsqueeze(1)).squeeze()
            t_scores = self.transitions[prev_tags, curr_tags]
                        
            # apply the mask
            e_scores = e_scores * is_valid
            t_scores = t_scores * is_valid
            
            scores += e_scores + t_scores
            
        # add transition from last tag to EOS for each sample in batch
        scores += self.transitions[last_tags, self.EOS_TAG_IDX]
        return scores
    
    # compute the partition function in log-space using forward algorithm
    def _compute_log_partition(self, emissions, mask):
        batch_size, seq_len, n_tags = emissions.shape
        
        # in the first step, SOS has all the scores
        alphas = self.transitions[self.SOS_TAG_IDX, :].unsqueeze(0) + emissions[:, 0]
        
        for i in range(1, seq_len):
            ## tensor[batch_size, n_tags] -> tensor[batch_size, 1, n_tags]
            e_scores = emissions[:, i].unsqueeze(1) 
            
            ## tensor[n_tags, n_tags] -> tensor[batch_size, n_tags, n_tags]
            t_scores = self.transitions.unsqueeze(0)
            
            ## tensor[batch_size, n_tags] -> tensor[batch_size, n_tags, 1]
            a_scores = alphas.unsqueeze(2)
            
            scores = e_scores + t_scores + a_scores
            new_alphas = torch.logsumexp(scores, dim = 1)
            
            # set alphas if the mask is valid, else keep current values
            is_valid = mask[:, i].unsqueeze(-1)
            alphas = is_valid * new_alphas + (1 - is_valid) * alphas
            
        # add scores for final transition
        last_transition = self.transitions[:, self.EOS_TAG_IDX]
        end_scores = alphas + last_transition.unsqueeze(0)
        
        # return log_sum_exp
        return torch.logsumexp(end_scores, dim = 1)
    
    # return a list of optimal tag sequence for each example in the batch
    def _viterbi_decode(self, emissions, mask):
        batch_size, seq_len, n_tags = emissions.shape
        
        # in the first iteration, SOS will have all the scores and then, the max
        alphas = self.transitions[self.SOS_TAG_IDX, :].unsqueeze(0) + emissions[:, 0]
        
        backpointers = []
        
        for i in range(1, seq_len):
            ## tensor[batch_size, n_tags] -> tensor[batch_size, 1, n_tags]
            e_scores = emissions[:, i].unsqueeze(1) 
            
            ## tensor[n_tags, n_tags] -> tensor[batch_size, n_tags, n_tags]
            t_scores = self.transitions.unsqueeze(0)
            
            ## tensor[batch_size, n_tags] -> tensor[batch_size, n_tags, 1]
            a_scores = alphas.unsqueeze(2)
            
            scores = e_scores + t_scores + a_scores
            
            # find the highest score and tag, instead of log_sum_exp
            max_scores, max_score_tags = torch.max(scores, dim = 1)
            
            # set alphas if the mask is valid, otherwise keep the current values
            is_valid = mask[:, i].unsqueeze(-1)
            alphas = is_valid * max_scores + (1 - is_valid) * alphas
            
            backpointers.append(max_score_tags.t())
            
        # add scores for final transition
        last_transition = self.transitions[:, self.EOS_TAG_IDX]
        end_scores = alphas + last_transition.unsqueeze(0)

        # get the final most probable score and the final most probable tag
        max_final_scores, max_final_tags = torch.max(end_scores, dim=1)

        # find the best sequence of labels for each sample in the batch
        best_sequences = []
        emission_lengths = mask.int().sum(dim=1)
        for i in range(batch_size):

            # recover the original sentence length for the i-th sample in the batch
            sample_length = emission_lengths[i].item()

            # recover the max tag for the last timestep
            sample_final_tag = max_final_tags[i].item()

            # limit the backpointers until the last but one
            # since the last corresponds to the sample_final_tag
            sample_backpointers = backpointers[: sample_length - 1]

            # follow the backpointers to build the sequence of labels
            sample_path = self._find_best_path(i, sample_final_tag, sample_backpointers)

            # add this path to the list of best sequences
            best_sequences.append(sample_path)

        return max_final_scores, best_sequences
    
    # auxiliary function to find the best path sequence for a specific example
    def _find_best_path(self, sample_id, best_tag, backpointers):
        ## backpointers: list[tensor[seq_len_i - 1, n_tags, batch_size]], seq_len_i is the length of the i-th sample of the batch
        
        # add the final best_tag to our best path
        best_path = [best_tag]

        # traverse the backpointers in backwards
        for backpointers_t in reversed(backpointers):

            # recover the best_tag at this timestep
            best_tag = backpointers_t[best_tag][sample_id].item()

            # append to the beginning of the list so we don't need to reverse it later
            best_path.insert(0, best_tag)

        return best_path
""")

In [18]:
file_path = "/kaggle/working/semantic-segmentation/model/Hier_BiLSTM_CRF.py"

with open(file_path, "w") as f:
    f.write("""
from model.submodels import *


    # Top-level module which uses a Hierarchical-LSTM-CRF to classify.
    
    # If pretrained = False, each example is represented as a sequence of sentences, which themselves are sequences of word tokens. Individual sentences are passed to LSTM_Sentence_Encoder to generate sentence embeddings. 
    # If pretrained = True, each example is represented as a sequence of fixed-length pre-trained sentence embeddings.
    
    # Sentence embeddings are then passed to LSTM_Emitter to generate emission scores, and finally CRF is used to obtain optimal tag sequence. 
    # Emission scores are fed to the CRF to generate optimal tag sequence.

class Hier_LSTM_CRF_Classifier(nn.Module):
    def __init__(self, n_tags, sent_emb_dim, sos_tag_idx, eos_tag_idx, pad_tag_idx, vocab_size = 0, word_emb_dim = 0, pad_word_idx = 0, pretrained = False, device = 'cuda'):
        super().__init__()
        
        self.emb_dim = sent_emb_dim
        self.pretrained = pretrained
        self.device = device
        self.pad_tag_idx = pad_tag_idx
        self.pad_word_idx = pad_word_idx
        
        # sentence encoder is not required for pretrained embeddings
        self.sent_encoder = LSTM_Sentence_Encoder(vocab_size, word_emb_dim, sent_emb_dim, device=self.device).to(self.device) if not self.pretrained else None
        self.emitter = LSTM_Emitter(n_tags, sent_emb_dim, sent_emb_dim, device=self.device).to(self.device)
        self.crf = CRF(n_tags, sos_tag_idx, eos_tag_idx, pad_tag_idx, device=self.device).to(self.device)
        
    
    def forward(self, x):
        batch_size = len(x)
        seq_lengths = [len(doc) for doc in x]
        max_seq_len = max(seq_lengths)
        
        if not self.pretrained: ## x: list[batch_size, sents_per_doc, words_per_sent]
            tensor_x = []
            for doc in x:
                sents = [torch.tensor(s, dtype = torch.long) for s in doc]
                sent_lengths = [len(s) for s in doc]
                
                ## list[sents_per_doc, words_per_sent] --> tensor[sents_per_doc, max_sent_len]
                sents = nn.utils.rnn.pad_sequence(sents, batch_first = True, padding_value = self.pad_word_idx).to(self.device)
                
                ## tensor[sents_per_doc, max_sent_len] --> tensor[sents_per_doc, sent_emb_dim]
                sents = self.sent_encoder(sents, sent_lengths)
        
                tensor_x.append(sents)
            
        else: ## x: list[batch_size, sents_per_doc, sent_emb_dim]
            tensor_x = [torch.tensor(doc, dtype = torch.float, requires_grad = True) for doc in x]
        
        ## list[batch_size, sents_per_doc, sent_emb_dim] --> tensor[batch_size, max_seq_len, sent_emb_dim]
        tensor_x = nn.utils.rnn.pad_sequence(tensor_x, batch_first = True).to(self.device)        
        
        self.mask = torch.zeros(batch_size, max_seq_len).to(self.device)
        for i, sl in enumerate(seq_lengths):
            self.mask[i, :sl] = 1	
        
        self.emissions = self.emitter(tensor_x)
        _, path = self.crf.decode(self.emissions, mask = self.mask)
        return path
    
    def _loss(self, y):
        ##  list[batch_size, sents_per_doc] --> tensor[batch_size, max_seq_len]
        tensor_y = [torch.tensor(doc, dtype = torch.long) for doc in y]
        tensor_y = nn.utils.rnn.pad_sequence(tensor_y, batch_first = True, padding_value = self.pad_tag_idx).to(self.device)
        
        nll = self.crf(self.emissions, tensor_y, mask = self.mask)
        return nll    
""")

In [19]:
file_path = "/kaggle/working/semantic-segmentation/prepare_data.py"

with open(file_path, "w") as f:
    f.write("""
import string
import os
import random

def prepare_folds(args):
    files = os.listdir(args.data_path)
    idx_order = [f[:-4] for f in files if f.endswith('.txt')]
    idx_order.sort()
    return idx_order

def prepare_data(idx_order, args, none_keep_ratio=0.3):
    x, y = [], []
    
    word2idx = {'<pad>': 0, '<unk>': 1}
    tag2idx  = {'<pad>': 0, '<start>': 1, '<end>': 2}

    for doc in idx_order:
        doc_x, doc_y = [], []
        try:
            with open(args.data_path + doc + '.txt') as fp:
                for sent in fp:
                    parts = sent.strip().split('\\t')
                    if len(parts) != 2:
                        continue
                    sent_x, sent_y = parts
                    sent_y = sent_y.strip()

                    # downsample None sentences
                    if sent_y == 'None' and random.random() > none_keep_ratio:
                        continue

                    if not args.pretrained:
                        sent_x = sent_x.strip().lower().translate(
                            str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
                        tokens = sent_x.split()
                        for tok in tokens:
                            if tok not in word2idx:
                                word2idx[tok] = len(word2idx)
                        sent_x = [word2idx[tok] for tok in tokens]
                    else:
                        sent_x = list(map(float, sent_x.strip().split()[:args.emb_dim]))

                    if sent_y not in tag2idx:
                        tag2idx[sent_y] = len(tag2idx)
                    sent_y = tag2idx[sent_y]

                    if sent_x:
                        doc_x.append(sent_x)
                        doc_y.append(sent_y)
        except Exception as e:
            print(f'Skipping {doc}: {e}')
            continue

        if doc_x:
            x.append(doc_x)
            y.append(doc_y)

    return x, y, word2idx, tag2idx

def prepare_data_inference(idx_order, args, sent2vec_model):
    x = []
    for doc in idx_order:
        doc_x = []
        try:
            with open(args.data_path + doc + '.txt') as fp:
                for sent in fp:
                    sent_x = sent.strip()
                    if not sent_x:
                        continue
                    if not args.pretrained:
                        sent_x = sent_x.lower().translate(
                            str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
                        sent_x = [args.word2idx.get(tok, args.word2idx['<unk>'])
                                  for tok in sent_x.split()]
                    else:
                        sent_x = sent2vec_model.embed_sentence(sent_x).flatten().tolist()[:args.emb_dim]
                    if sent_x:
                        doc_x.append(sent_x)
        except Exception as e:
            print(f'Skipping {doc}: {e}')
            continue
        if doc_x:
            x.append(doc_x)
    return x
""")

In [20]:
file_path = "/kaggle/working/semantic-segmentation/run.py"

with open(file_path, "w") as f:
    f.write("""
import argparse
import torch
from model.Hier_BiLSTM_CRF import *
from prepare_data import *
from train import *

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument('--pretrained', default = False, type = bool, help = 'Whether the model uses pretrained sentence embeddings or not')
    parser.add_argument('--data_path', default = 'data/text/', type = str, help = 'Folder to store the annotated text files')
    parser.add_argument('--save_path', default = 'saved/', type = str, help = 'Folder where predictions and models will be saved')
    parser.add_argument('--cat_path', default = 'categories.txt', type = str, help = 'Path to file containing category details')     
    parser.add_argument('--dataset_size', default = 50, type = int, help = 'Total no. of docs')
    parser.add_argument('--num_folds', default = 5, type = int, help = 'No. of folds to divide the dataset into')
    parser.add_argument('--device', default = 'cuda', type = str, help = 'cuda / cpu')
    
    parser.add_argument('--batch_size', default = 4, type = int)
    parser.add_argument('--print_every', default = 5, type = int, help = 'Epoch interval after which validation macro f1 and loss will be printed')
    parser.add_argument('--lr', default = 0.01, type = float, help = 'Learning Rate')
    parser.add_argument('--reg', default = 0, type = float, help = 'L2 Regularization')
    parser.add_argument('--emb_dim', default = 200, type = int, help = 'Sentence embedding dimension')     
    parser.add_argument('--word_emb_dim', default = 100, type = int, help = 'Word embedding dimension, applicable only if pretrained = False')
    parser.add_argument('--epochs', default = 300, type = int)

    parser.add_argument('--val_fold', default = 'cross', type = str, help = 'Fold number to be used as validation, use cross for num_folds cross validation')
    args = parser.parse_args()

    args.device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Preparing data ...', end = ' ')
    idx_order = prepare_folds(args)
    idx_order = idx_order[:1000]
    x, y, word2idx, tag2idx = prepare_data(idx_order, args)
    # x, y = x[:1000], y[:1000]
    print('Done')

    print('Vocabulary size:', len(word2idx))
    print('#Tags:', len(tag2idx))

    # Dump word2idx and tag2idx
    with open(args.save_path + 'word2idx.json', 'w') as fp:
        json.dump(word2idx, fp)
    with open(args.save_path + 'tag2idx.json', 'w') as fp:
        json.dump(tag2idx, fp)

    print('Initializing model ...', end = ' ')  
    model = Hier_LSTM_CRF_Classifier(
        len(tag2idx),
        args.emb_dim,
        tag2idx['<start>'],
        tag2idx['<end>'],
        tag2idx['<pad>'],
        vocab_size = len(word2idx),
        word_emb_dim = args.word_emb_dim,
        pretrained = args.pretrained,
        device = args.device
    ).to(args.device)
    print('Done')
    
    print('Training model ...')        
    learn(model, x, y, tag2idx, 0, args)

if __name__ == '__main__':
    main()
""")

In [21]:
file_path = "/kaggle/working/semantic-segmentation/infer.py"

with open(file_path, "w") as f:
    f.write("""
import argparse
import sent2vec
import os

from model.Hier_BiLSTM_CRF import *
from prepare_data import *
from train import *

def main():
    parser = argparse.ArgumentParser(description = 'Infer tags for unannotated files')

    parser.add_argument('--pretrained', default = False, type = bool, help = 'Whether the model uses pretrained sentence embeddings or not')
    parser.add_argument('--data_path', default = 'infer/data/', type = str, help = 'Folder to store the unannotated text files')
    parser.add_argument('--model_path', default = 'infer/model.tar', type = str, help = 'Path to trained Hierarchical BiLSTM CRF Model')
    parser.add_argument('--sent2vec_model_path', default = 'infer/sent2vec_model.bin', type = str, help = 'Path to trained sent2vec model, applicable only if pretrained = True')
    parser.add_argument('--save_path', default = 'infer/predictions.txt', type = str, help = 'Path to file where predictions will be saved')
    parser.add_argument('--word2idx_path', default = 'infer/word2idx.json', type = str, help = 'Path to word2idx dict created during training model')
    parser.add_argument('--tag2idx_path', default = 'infer/tag2idx.json', type = str, help = 'Path to tag2idx dict created during training model')
    parser.add_argument('--emb_dim', default = 200, type = int, help = 'Sentence embedding dimension')
    parser.add_argument('--word_emb_dim', default = 100, type = int, help = 'Word embedding dimension, applicable only if pretrained = False')
    parser.add_argument('--device', default = 'cuda', type = str, help = 'cuda / cpu')
    
    args = parser.parse_args()

    with open(args.word2idx_path) as fp:
        args.word2idx = json.load(fp)

    with open(args.tag2idx_path) as fp:
        args.tag2idx = json.load(fp)

    if args.pretrained:
        print('Loading pretrained sent2vec model ...', end = ' ', flush = True)
        sent2vec_model = sent2vec.Sent2vecModel()
        sent2vec_model.load_model(args.sent2vec_model_path)
        print('Done')

    else:
        sent2vec_model = None

    print('Preparing data ...', end = ' ', flush = True)
    idx_order = [os.fsdecode(x)[:-4] for x in os.listdir(os.fsencode(args.data_path))
             if os.fsdecode(x).endswith('.txt') and not os.fsdecode(x).startswith('_')]
    x = prepare_data_inference(idx_order, args, sent2vec_model)
    print('Done')

    print('Loading model ...', end = ' ', flush = True)

    ckpt = torch.load(args.model_path, weights_only=False, map_location='cpu')

    model = Hier_LSTM_CRF_Classifier(len(args.tag2idx), args.emb_dim, args.tag2idx['<start>'], args.tag2idx['<end>'], args.tag2idx['<pad>'], vocab_size = len(args.word2idx), word_emb_dim = args.word_emb_dim, pretrained = args.pretrained, device = args.device).to(args.device)

    model.load_state_dict(ckpt['state_dict'])    

    print('Done')

    pred = infer_step(model, x)

    idx2tag = {v: k for (k, v) in args.tag2idx.items()}

    print('Saving predictions ...', end = ' ', flush = True)    
    with open(args.save_path, 'w') as fp:
        for idx, doc in enumerate(idx_order):
            print(doc, end = "\t", file = fp)
            p = list(map(lambda x: idx2tag[x], pred[idx]))
            print(*p, sep = ',', file = fp)
    print('Done')

if __name__ == '__main__':
	main()
""")

In [22]:
!python run.py --epochs 100 --batch_size 4 --print_every 1

Preparing data ... Done
Vocabulary size: 55465
#Tags: 10
Initializing model ... Done
Training model ...
  EPOCH     Tr_LOSS   Tr_F1    Val_LOSS  Val_F1
-----------------------------------------------------------
      1     543.174   0.406     363.892   0.491
      2     329.443   0.499     333.537   0.526
      3     295.130   0.573     316.095   0.592
      4     281.483   0.604     312.447   0.626
      5     270.819   0.643     300.778   0.644
      6     264.986   0.659     301.981   0.646
      7     263.452   0.662     321.396   0.613
      8     263.013   0.665     315.521   0.597
      9     259.953   0.671     304.015   0.637
     10     255.865   0.687     307.550   0.605
     11     255.907   0.697     316.701   0.618
     12     253.021   0.697     317.853   0.603
     13     253.291   0.696     308.879   0.652
     14     249.135   0.712     323.949   0.637
     15     250.959   0.701     326.326   0.612
     16     249.851   0.711     331.892   0.621
     17     249.420 

In [23]:
!ls

categories.txt	infer	  LICENSE  prepare_data.py  README.md  saved
data		infer.py  model    __pycache__	    run.py     train.py


In [24]:
!cp saved/model_state.tar infer/model.tar
!cp saved/word2idx.json infer/
!cp saved/tag2idx.json infer/

In [25]:
!pwd

/kaggle/working/semantic-segmentation


In [26]:
!wget -O infer/sent2vec_model.bin "http://cse.iitkgp.ac.in/~saptarshi/models/sent2vec.bin"

--2026-04-24 14:38:34--  http://cse.iitkgp.ac.in/~saptarshi/models/sent2vec.bin
Resolving cse.iitkgp.ac.in (cse.iitkgp.ac.in)... 203.110.245.250
Connecting to cse.iitkgp.ac.in (cse.iitkgp.ac.in)|203.110.245.250|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://cse.iitkgp.ac.in/~saptarshi/models/sent2vec.bin [following]
--2026-04-24 14:38:35--  https://cse.iitkgp.ac.in/~saptarshi/models/sent2vec.bin
Connecting to cse.iitkgp.ac.in (cse.iitkgp.ac.in)|203.110.245.250|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2252249083 (2.1G) [application/octet-stream]
Saving to: ‘infer/sent2vec_model.bin’

infer/sent2vec_mode 100%[===================>]   2.10G  9.49MB/s    in 3m 58s  

2026-04-24 14:42:34 (9.04 MB/s) - ‘infer/sent2vec_model.bin’ saved [2252249083/2252249083]



In [27]:
!ls

categories.txt	infer	  LICENSE  prepare_data.py  README.md  saved
data		infer.py  model    __pycache__	    run.py     train.py


In [28]:
%cd ..

/kaggle/working


In [29]:
!git clone https://github.com/epfml/sent2vec.git
%cd /kaggle/working/sent2vec
!pip install .
%cd /kaggle/working/semantic-segmentation

Cloning into 'sent2vec'...
remote: Enumerating objects: 425, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 425 (delta 9), reused 4 (delta 1), pack-reused 403 (from 1)
Receiving objects: 100% (425/425), 447.46 KiB | 6.88 MiB/s, done.
Resolving deltas: 100% (261/261), done.
/kaggle/working/sent2vec
Processing /kaggle/working/sent2vec
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for sent2vec: filename=sent2vec-0.0.0-cp312-cp312-linux_x86_64.whl size=1659000 sha256=4605d8c7c9c91d53abd5e1a78755dbd01dc4346f1218a81df0660cca21cdcc5d
  Stored in directory: /tmp/pip-ephem-wheel-cache-u19z6k6j/wheels/53/65/15/bc6b844070f84491a73d70035c66bf78a78158f776442a24c0
Successfully built sent2vec
/kaggle/working/semantic-segmentation


In [30]:
import os
print(os.getcwd())  
print(os.listdir('infer/'))  
print(len(os.listdir('infer/data/')), "documents")

/kaggle/working/semantic-segmentation
['data', 'model.tar', 'sent2vec_model.bin', 'tag2idx.json', 'word2idx.json']
1 documents


In [31]:
test_data = datasets['test_queries']
print(test_data)

import os

shutil.rmtree('infer/data/', ignore_errors=True)
os.makedirs('infer/data/')

os.makedirs('infer/data/', exist_ok=True)

for sample in test_data:  
    doc_id = sample['id']
    sentences = sample['text']
    
    with open(f'infer/data/{doc_id}.txt', 'w') as f:
        for sent in sentences:
            f.write(sent.strip() + '\n')

print(f"Written {len(test_data)} files")

test_file = os.listdir('infer/data/')[0]
with open(f'infer/data/{test_file}') as f:
    lines = f.readlines()
print(f"Sample file '{test_file}' has {len(lines)} sentences")
print(f"First sentence: {lines[0].strip()}")

import os
print(os.getcwd())  
print(os.listdir('infer/'))  
print(len(os.listdir('infer/data/')), "documents")

import os, shutil
import torch
import gc

files = [f for f in os.listdir('infer/data/') if f.endswith('.txt')]
batch_size = 50  
batches = [files[i:i+batch_size] for i in range(0, len(files), batch_size)]
print(f"Total files: {len(files)}, Total batches: {len(batches)}")

open('infer/test_queries.txt', 'w').close()

for batch_num, batch in enumerate(batches):
    print(f"\nProcessing batch {batch_num+1}/{len(batches)} ({len(batch)} docs)...")
    
    # Create temp folder with just this batch
    shutil.rmtree('infer/data_batch/', ignore_errors=True)
    os.makedirs('infer/data_batch/')
    for f in batch:
        shutil.copy(f'infer/data/{f}', f'infer/data_batch/{f}')
    
    # Clear GPU memory before each batch
    torch.cuda.empty_cache()
    gc.collect()
    
    # Run inference
    os.system('python infer.py --data_path infer/data_batch/ --save_path infer/predictions_batch.txt --device cpu')
    
    # Append batch predictions to main file
    with open('infer/test_queries.txt', 'a') as main_f:
        with open('infer/predictions_batch.txt', 'r') as batch_f:
            main_f.write(batch_f.read())

    print(f"Batch {batch_num+1} done.")

print("\nAll done!")
print(f"Total predictions: {sum(1 for _ in open('infer/test_queries.txt'))}")

Dataset({
    features: ['id', 'text', 'relevant_candidates'],
    num_rows: 237
})
Written 237 files
Sample file '0001355522.txt' has 44 sentences
First sentence: PETITIONER: <NAME> AND ANOTHER Vs. RESPONDENT: THE STATE OF MADHYA PRADESH. DATE OF JUDGMENT: 14/05/1954 BENCH: <NAME>, <NAME> (CJ) BENCH: <NAME>, <NAME> (CJ) MUKHERJEA, <NAME>, VIVIAN BHAGWATI, <NAME>, T.L. VENKATARAMA CITATION: 1954 AIR 465 1955 SCR 313 CITATOR INFO : R 1957 SC 478 (11) R 1957 SC 510 (14) RF 1957 SC 896 (12) R 1960 SC 475 (4,9,13,16) RF 1961 SC 4 (4,15) R 1961 SC 705 (17) R 1961 SC1602 (12) F 1961 SC1731 (13) RF 1964 SC 381 (38) R 1965 SC1107 (60) R 1966 SC1788 (10,14) RF 1967 SC 212 (27) RF 1967 SC 669 (21) RF 1968 SC1232 (15,53,82,95) RF 1970 SC 564 (185) RF 1973 SC 106 (147) RF 1973 SC1461 (227,450,566,1847,1848,1998) R 1974 SC 366 (56) E&D 1974 SC 543 (13) R 1974 SC1660 (18) R 1978 SC 851 (39) RF 1978 SC1296 (12) E 1980 SC 350 (4) R 1982 SC1126 (10,11) RF 1983 SC1019 (29,30) F 1987 SC1802 (9) R 1990 SC

In [35]:
test_data = datasets['train_queries']
print(test_data)

import os

shutil.rmtree('infer/data/', ignore_errors=True)
os.makedirs('infer/data/')

os.makedirs('infer/data/', exist_ok=True)

for sample in test_data:  
    doc_id = sample['id']
    sentences = sample['text']
    
    with open(f'infer/data/{doc_id}.txt', 'w') as f:
        for sent in sentences:
            f.write(sent.strip() + '\n')

print(f"Written {len(test_data)} files")

test_file = os.listdir('infer/data/')[0]
with open(f'infer/data/{test_file}') as f:
    lines = f.readlines()
print(f"Sample file '{test_file}' has {len(lines)} sentences")
print(f"First sentence: {lines[0].strip()}")

import os
print(os.getcwd())  
print(os.listdir('infer/'))  
print(len(os.listdir('infer/data/')), "documents")

import os, shutil
import torch
import gc

files = [f for f in os.listdir('infer/data/') if f.endswith('.txt')]
batch_size = 50  
batches = [files[i:i+batch_size] for i in range(0, len(files), batch_size)]
print(f"Total files: {len(files)}, Total batches: {len(batches)}")

open('infer/train_queries.txt', 'w').close()

for batch_num, batch in enumerate(batches):
    print(f"\nProcessing batch {batch_num+1}/{len(batches)} ({len(batch)} docs)...")
    
    # Create temp folder with just this batch
    shutil.rmtree('infer/data_batch/', ignore_errors=True)
    os.makedirs('infer/data_batch/')
    for f in batch:
        shutil.copy(f'infer/data/{f}', f'infer/data_batch/{f}')
    
    # Clear GPU memory before each batch
    torch.cuda.empty_cache()
    gc.collect()
    
    # Run inference
    os.system('python infer.py --data_path infer/data_batch/ --save_path infer/predictions_batch.txt --device cpu')
    
    # Append batch predictions to main file
    with open('infer/train_queries.txt', 'a') as main_f:
        with open('infer/predictions_batch.txt', 'r') as batch_f:
            main_f.write(batch_f.read())

    print(f"Batch {batch_num+1} done.")

print("\nAll done!")
print(f"Total predictions: {sum(1 for _ in open('infer/train_queries.txt'))}")

Dataset({
    features: ['id', 'text', 'relevant_candidates'],
    num_rows: 827
})
Written 827 files
Sample file '0001111557.txt' has 27 sentences
First sentence: PETITIONER: C.I.T. (CENTRAL), <NAME> Vs. RESPONDENT: <ORG>., KODIALBALL, MANGALORE DATE OF JUDGMENT15/07/1986 BENCH: <NAME>, R.S. BENCH: <NAME>, R.S. MUKHARJI, <NAME> (J) CITATION: 1986 AIR 1727 1986 SCR (3) 166 1986 SCC (3) 538 JT 1986 302 1986 SCALE (2)29 ACT: Income Tax Act , 1961-s. 80E-Profit and gains-Priority industries-deductions in respect of-how determined. HEADNOTE: The assessee-company is engaged in the manufacture of autombile spares. During the previous year relevant to the assessment year 1966-67, the assessee also commenced the manufacturing of alloy steels. Both the industries are included in the Fifth Schedule to the Income Tax Act , 1961. The assessee sustained a loss in the alloys steel industry during the previous years relevant to the assessment years 1966-67 and 1967-68. It claimed a loss in the sum of

In [36]:
test_data = datasets['dev_queries']
print(test_data)

import os

shutil.rmtree('infer/data/', ignore_errors=True)
os.makedirs('infer/data/')

os.makedirs('infer/data/', exist_ok=True)

for sample in test_data:  
    doc_id = sample['id']
    sentences = sample['text']
    
    with open(f'infer/data/{doc_id}.txt', 'w') as f:
        for sent in sentences:
            f.write(sent.strip() + '\n')

print(f"Written {len(test_data)} files")

test_file = os.listdir('infer/data/')[0]
with open(f'infer/data/{test_file}') as f:
    lines = f.readlines()
print(f"Sample file '{test_file}' has {len(lines)} sentences")
print(f"First sentence: {lines[0].strip()}")

import os
print(os.getcwd())  
print(os.listdir('infer/'))  
print(len(os.listdir('infer/data/')), "documents")

import os, shutil
import torch
import gc

files = [f for f in os.listdir('infer/data/') if f.endswith('.txt')]
batch_size = 50  
batches = [files[i:i+batch_size] for i in range(0, len(files), batch_size)]
print(f"Total files: {len(files)}, Total batches: {len(batches)}")

open('infer/val_queries.txt', 'w').close()

for batch_num, batch in enumerate(batches):
    print(f"\nProcessing batch {batch_num+1}/{len(batches)} ({len(batch)} docs)...")
    
    # Create temp folder with just this batch
    shutil.rmtree('infer/data_batch/', ignore_errors=True)
    os.makedirs('infer/data_batch/')
    for f in batch:
        shutil.copy(f'infer/data/{f}', f'infer/data_batch/{f}')
    
    # Clear GPU memory before each batch
    torch.cuda.empty_cache()
    gc.collect()
    
    # Run inference
    os.system('python infer.py --data_path infer/data_batch/ --save_path infer/predictions_batch.txt --device cpu')
    
    # Append batch predictions to main file
    with open('infer/val_queries.txt', 'a') as main_f:
        with open('infer/predictions_batch.txt', 'r') as batch_f:
            main_f.write(batch_f.read())

    print(f"Batch {batch_num+1} done.")

print("\nAll done!")
print(f"Total predictions: {sum(1 for _ in open('infer/val_queries.txt'))}")

Dataset({
    features: ['id', 'text', 'relevant_candidates'],
    num_rows: 118
})
Written 118 files
Sample file '0146351380.txt' has 181 sentences
First sentence: Reportable IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 223 OF 2008 Rattiram & Ors. .............Appellant Versus State of M. P. Through Inspector of Police .........Respondent WITH CRIMINAL APPEAL NO. 458 OF 2008 Satyanarayan & ors. ............Appellant Versus The State of Madhya Pradesh Through Incharge, Police Station Cantt. .........Respondent J U D G M E N T
/kaggle/working/semantic-segmentation
['predictions_batch.txt', 'train_docs.txt', 'data', 'model.tar', 'test_queries.txt', 'data_batch', 'test_docs.txt', 'train_queries.txt', 'val_docs.txt', 'sent2vec_model.bin', 'tag2idx.json', 'word2idx.json']
118 documents
Total files: 118, Total batches: 3

Processing batch 1/3 (50 docs)...
Preparing data ... Done
Loading model ... Done
Saving predictions ... Done
Batch 1 done.

Processing 